# 10 - Orquestador del Pipeline (ejecución desde un notebook)

Trabajo Práctico - Pipeline ETL end-to-end (Bronze -> Silver / DQL -> Gold)

El pipeline tiene **dos formas de orquestación**, que respetan las mismas dependencias:

1. **Job de Databricks** (`job_pipeline_datos_empresas.json`, en esta misma carpeta). Es la orquestación principal: cada notebook es una tarea, el Job muestra el grafo de dependencias (la imagen del pipeline que se entrega) y se puede programar.
2. **Este notebook**: ejecuta el mismo flujo con `dbutils.notebook.run`. Lo usa `11_Prueba_Idempotencia` para correr el pipeline varias veces seguidas, y sirve para ejecutar todo a mano sin crear el Job.

Este notebook vive en `Trabajo Practico 2 - Orquestacion e Idempotencia/`, junto con `09_Auditoria_Ejecucion`, `11_Prueba_Idempotencia`, `12_Analisis_Preguntas_Negocio` y el Job JSON. Los notebooks `00` a `05` (Silver) viven en `Trabajo Practico 2 - Capa Silver/`, los `06` a `08` (Gold) en `Trabajo Practico 2 - Capa Gold/`, y los de ingesta Bronze (`02` a `05`) en `Trabajo Practico 1 - Arquitectura Inicial (Capa Bronze)/`.

## Widget `ejecutar_bronze`

Por defecto (`si`) el pipeline arranca re-ingestando las 4 fuentes Bronze (Etapa 0). Si Bronze ya está al día y solo se quiere reprocesar Silver/Gold, poner `no` salta esa etapa sin afectar las demás.

## Ejecución notebook por notebook (secuencial, no en paralelo)

Cada notebook se ejecuta uno a la vez con `dbutils.notebook.run`, incluso dentro de una misma etapa. Lanzar varios notebooks a la vez contra el mismo cómputo Serverless lo satura y produce timeouts ("Databricks is unresponsive"), así que aunque las etapas documentan qué tareas no dependen entre sí, el notebook las corre en secuencia para no sobrecargar el cómputo.

## Grafo de dependencias

```
(Etapa 0, opcional) Ingesta Bronze: RUES, TRM, CIIU, SECOP
        |
        v
01_Configuracion_Pipeline
        |
        +--> 02_Silver_CIIU --> 03_Silver_RUES --> 06_Gold_Dim_Empresas --+
        +--> 04_Silver_SECOP ---------------------------------------------+--> 07_Gold_Fact_Contratos --> 08_Gold_Agg_Indicadores_Mes --> 09_Auditoria_Ejecucion
        `--> 05_Silver_TRM -----------------------------------------------+
```

Dentro de cada etapa las tareas no dependen entre sí (podrían ir en paralelo), pero se ejecutan una por una. Una etapa empieza solo cuando terminó bien la anterior. Si una tarea falla, el pipeline se detiene y no se construye Gold con datos incompletos.

In [0]:
%python
import json
from datetime import datetime

dbutils.widgets.text("id_ejecucion", "", "Id de ejecución (vacío = automático)")
ID_EJECUCION = dbutils.widgets.get("id_ejecucion").strip() or f"orquestador_{datetime.now():%Y%m%d_%H%M%S}"

dbutils.widgets.dropdown("ejecutar_bronze", "si", ["si", "no"], "Re-ingestar Bronze")
EJECUTAR_BRONZE = dbutils.widgets.get("ejecutar_bronze") == "si"

TIMEOUT_SEGUNDOS = 3600

RUTA_BRONZE = "../Trabajo Practico 1 - Arquitectura Inicial (Capa Bronze)"
RUTA_SILVER = "../Trabajo Practico 2 - Capa Silver"
RUTA_GOLD = "../Trabajo Practico 2 - Capa Gold"

ETAPAS = [
    [f"{RUTA_BRONZE}/02_Ingesta_Bronze_RUES", f"{RUTA_BRONZE}/03_Ingesta_Bronze_TRM",
     f"{RUTA_BRONZE}/04_Ingesta_Bronze_CIIU", f"{RUTA_BRONZE}/05_Ingesta_Bronze_SECOP"],
    [f"{RUTA_SILVER}/01_Configuracion_Pipeline"],
    [f"{RUTA_SILVER}/02_Silver_CIIU", f"{RUTA_SILVER}/04_Silver_SECOP", f"{RUTA_SILVER}/05_Silver_TRM"],
    [f"{RUTA_SILVER}/03_Silver_RUES"],
    [f"{RUTA_GOLD}/06_Gold_Dim_Empresas"],
    [f"{RUTA_GOLD}/07_Gold_Fact_Contratos"],
    [f"{RUTA_GOLD}/08_Gold_Agg_Indicadores_Mes"],
    ["09_Auditoria_Ejecucion"],
]

if not EJECUTAR_BRONZE:
    ETAPAS = ETAPAS[1:]

print(f"id_ejecucion = {ID_EJECUCION} | ejecutar_bronze = {EJECUTAR_BRONZE}")

In [0]:
%python
def ejecutar(notebook: str) -> dict:
    inicio = datetime.now()
    ruta = notebook if notebook.startswith("..") else f"./{notebook}"
    salida = dbutils.notebook.run(ruta, TIMEOUT_SEGUNDOS, {"id_ejecucion": ID_EJECUCION})
    segundos = (datetime.now() - inicio).total_seconds()
    print(f"  OK {notebook} ({segundos:.0f} s)")
    return {"notebook": notebook, "segundos": round(segundos), "salida": json.loads(salida) if salida else None}


resultados = []
inicio_total = datetime.now()

for numero, etapa in enumerate(ETAPAS, start=1):
    print(f"Etapa {numero}/{len(ETAPAS)}: {', '.join(etapa)}")
    for nb in etapa:
        resultados.append(ejecutar(nb))

print(f"\nPipeline completo en {(datetime.now() - inicio_total).total_seconds():.0f} s")

In [0]:
%python
display(spark.createDataFrame([(r["notebook"], r["segundos"]) for r in resultados], "notebook string, segundos long"))

dbutils.notebook.exit(json.dumps({"id_ejecucion": ID_EJECUCION, "estado": "OK"}))